<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part B: Statistical Forecasting</h2>
<h2>Notebook B02: ARIMA Models</h2>
</div>

Exponential smoothing describes a series through components you choose in advance: a level, a trend, a
season. ARIMA takes the opposite route. It assumes almost nothing about shape, and instead models the
series through its own past values and its own past errors, letting the correlation structure of the data
decide what form the model takes.

That makes it more flexible and considerably less forgiving. You have to make the series stationary
first, and you have to choose the orders. This notebook works through both, then adds seasonality and,
finally, outside information.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [Stationarity and Differencing](#2.-Stationarity-and-Differencing)
3. [AR and MA: The Building Blocks](#3.-AR-and-MA:-The-Building-Blocks)
4. [Reading Orders from ACF and PACF](#4.-Reading-Orders-from-ACF-and-PACF)
5. [ARIMA and SARIMA](#5.-ARIMA-and-SARIMA)
6. [Searching for the Orders](#6.-Searching-for-the-Orders)
7. [SARIMAX: Using What You Already Know](#7.-SARIMAX:-Using-What-You-Already-Know)
8. [Where ARIMA Fits](#8.-Where-ARIMA-Fits)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss

import nb_config

sns.set_theme(style="whitegrid")

Same series and same split as Notebooks [A05](./A05_Forecasting_baselines.ipynb),
[A06](./A06_Evaluating_models.ipynb) and [B01](./B01_Exponential_smoothing_models.ipynb), so every number
below can be compared with what came before. Section 7 also uses the Rossmann sales data from
[A04](./A04_Handling_outliers.ipynb).

Two targets to keep in mind: the seasonal naive baseline at **1.74 °C**, and Holt-Winters at
**1.22 °C**.

In [ ]:
series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12

train = series.iloc[:-TEST_MONTHS]
test = series.iloc[-TEST_MONTHS:]


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


SEASONAL_NAIVE_MAE = 1.74   # from Notebook A05
HOLT_WINTERS_MAE = 1.22     # from Notebook B01

print(f"Train: {len(train)} months, Test: {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Stationarity-and-Differencing">2. Stationarity and Differencing</h3>
</div>

ARIMA is built for **stationary** series: ones whose statistical behaviour does not depend on when you
look. Constant mean, constant variance, and a correlation between two points that depends only on how far
apart they are. A series with a trend is not stationary, because its mean moves. A series with a season is
not stationary either, because its behaviour depends on the month.

Two tests are standard, and they are not the same test with different names:

- **ADF** (Augmented Dickey-Fuller) has non-stationarity as its null hypothesis. A **small p-value means
  stationary**.
- **KPSS** has stationarity as its null. A **small p-value means non-stationary**.

They point in opposite directions, which is exactly why you run both.

In [ ]:
def check_stationarity(values, label):
    """Run both tests and print what each concludes."""
    values = pd.Series(values).dropna()

    with warnings.catch_warnings():
        # Both tests warn when the p-value falls off the end of their lookup table
        warnings.simplefilter("ignore")
        adf_stat, adf_p = adfuller(values, autolag="AIC")[:2]
        kpss_stat, kpss_p = kpss(values, regression="c", nlags="auto")[:2]

    print(f"--- {label} ---")
    print(f"  ADF   statistic={adf_stat:7.3f}  p={adf_p:.4f}  -> "
          f"{'stationary' if adf_p < 0.05 else 'NOT stationary'}")
    print(f"  KPSS  statistic={kpss_stat:7.3f}  p={kpss_p:.4f}  -> "
          f"{'NOT stationary' if kpss_p < 0.05 else 'stationary'}")
    print()


check_stationarity(train, "Raw series")

The two tests **disagree**, and that disagreement is informative rather than annoying.

ADF looks for a unit root, the signature of a random walk that wanders off and never returns. Temperature
does not do that: it always comes back, so ADF sees a series that reverts to its mean and reports
stationarity. KPSS asks the broader question of whether the statistical behaviour is constant over time,
and the answer is plainly no, because it depends heavily on the month.

When ADF says stationary and KPSS says otherwise, the usual culprit is seasonality or a slow trend:
structure that is not a unit root but is still time dependence. The fix is **differencing**, and for a
seasonal series it is *seasonal* differencing: subtract the value from one full cycle ago.

In [ ]:
seasonally_differenced = train.diff(SEASON_LENGTH).dropna()

check_stationarity(seasonally_differenced, f"After differencing at lag {SEASON_LENGTH}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(train["1990":], color="steelblue", linewidth=0.9)
axes[0].set_title("Raw series", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")

axes[1].plot(seasonally_differenced["1990":], color="seagreen", linewidth=0.9)
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].set_title(f"Differenced at lag {SEASON_LENGTH}", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Change vs 12 months earlier (°C)")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Now both tests agree, and the plot shows why: the seasonal swing is gone and what remains oscillates
around zero with roughly constant spread.

In ARIMA notation this is **D = 1** with **m = 12**: one seasonal difference at a cycle length of twelve.
We did not need an ordinary difference (**d = 0**), because once the season is removed there is no
remaining trend strong enough to matter over this record.

A warning worth taking seriously: **differencing is not free**. Each difference throws away information
and inflates the variance of the noise. Difference only as much as the tests require, never "to be safe".

**Exercise.** Apply an ordinary first difference (`train.diff()`) instead of a seasonal one and run the tests again. Do they agree? Plot the result: what has first differencing done to the seasonal pattern, and why is it the wrong tool here?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-AR-and-MA:-The-Building-Blocks">3. AR and MA: The Building Blocks</h3>
</div>

ARIMA is assembled from two ideas, each of which is a model in its own right.

**AR(p), autoregressive.** Predict the next value from a weighted sum of the last $p$ values:

$$y_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \epsilon_t$$

The series remembers its own past. This is what captures momentum and mean reversion.

**MA(q), moving average.** Predict the next value from the last $q$ *forecast errors*:

$$y_t = \mu + \epsilon_t + \theta_1\epsilon_{t-1} + \dots + \theta_q\epsilon_{t-q}$$

The name is unfortunate, since this has nothing to do with the moving averages of Notebook A05. Here the
series remembers its own *shocks*: a surprise last month still echoes in this month's value.

**ARMA(p,q)** uses both. Fitted on the raw series, with no differencing and no seasonal terms, these are
the models that ARIMA reduces to.

In [ ]:
building_blocks = {
    "AR(1)": (1, 0, 0),
    "AR(2)": (2, 0, 0),
    "MA(2)": (0, 0, 2),
    "ARMA(2,2)": (2, 0, 2),
}

rows = []
for name, order in building_blocks.items():
    model = ARIMA(train, order=order).fit()
    rows.append({
        "Model": name,
        "order": str(order),
        "AIC": model.aic,
        "Test MAE": mean_absolute_error(test, model.forecast(TEST_MONTHS)),
    })

pd.DataFrame(rows).round(2)

None of them is close to the 1.74 °C baseline, which is no surprise: none has any way to express "this is
February". What they can do is exploit short-range correlation, and ARMA(2,2) gets as far as 2.33 °C on
that alone, which is a good deal better than the 6.04 °C of a plain mean forecast.

The lesson matches Notebook B01 exactly. The missing ingredient is the season, and no amount of tuning
$p$ and $q$ will substitute for it.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Reading-Orders-from-ACF-and-PACF">4. Reading Orders from ACF and PACF</h3>
</div>

How do you pick $p$ and $q$? The classical answer is to read them off the ACF and PACF plots from
Notebook [A02](./A02_Basic_plotting.ipynb), computed on the **stationary** version of the series.

The two rules, which are worth memorising:

| Pattern | Suggests |
|---|---|
| ACF decays gradually, PACF **cuts off** after lag $p$ | **AR(p)** |
| ACF **cuts off** after lag $q$, PACF decays gradually | **MA(q)** |
| Both decay gradually | **ARMA**, orders unclear |

"Cuts off" means the spikes drop inside the shaded confidence band and stay there. The same rules apply
at seasonal lags: look at lags 12, 24, 36 for monthly data to read the seasonal orders $P$ and $Q$.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

plot_acf(seasonally_differenced, lags=30, ax=axes[0], color="steelblue",
         vlines_kwargs={"colors": "steelblue"})
axes[0].set_title("ACF of the seasonally differenced series", fontsize=13, fontweight="bold")

plot_pacf(seasonally_differenced, lags=30, ax=axes[1], color="steelblue",
          vlines_kwargs={"colors": "steelblue"})
axes[1].set_title("PACF of the seasonally differenced series", fontsize=13, fontweight="bold")

for ax in axes:
    ax.set_xlabel("Lag (months)")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Read it in two parts.

**The short lags.** The ACF decays from 0.21 at lag 1 through 0.09 and 0.05, while the PACF has one spike
at lag 1 and then drops inside the band. Gradual ACF, sharp PACF cut-off: that is the **AR(1)**
signature, so $p = 1$, $q = 0$.

**The seasonal lags.** There is a single large negative spike at lag 12 in the ACF (-0.50) and nothing of
comparable size at 24 or 36, where it sits at the edge of the band, while the PACF decays across 12, 24 and 36 (-0.50, -0.25, and on). Sharp ACF cut-off,
decaying PACF, both at the seasonal lags: that is the **MA** signature applied seasonally, so $Q = 1$ and
$P = 0$.

Putting it together, the plots suggest **SARIMA(1,0,0)(0,1,1,12)**. Let us fit exactly that, along with
the AR(1) model on its own for contrast.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-ARIMA-and-SARIMA">5. ARIMA and SARIMA</h3>
</div>

**ARIMA(p,d,q)** is ARMA with differencing folded in: fit ARMA to the series after differencing it $d$
times, and undo the differencing when forecasting.

**SARIMA(p,d,q)(P,D,Q,m)** adds a second set of the same three orders operating at the seasonal lag, with
$m$ the length of the cycle. The seasonal terms do at lag 12 what the ordinary terms do at lag 1.

`statsmodels` spells both with the same `ARIMA` class; a non-zero `seasonal_order` is what makes it a
SARIMA.

In [ ]:
hand_read = ARIMA(
    train,
    order=(1, 0, 0),
    seasonal_order=(0, 1, 1, SEASON_LENGTH),
).fit()

hand_read_forecast = hand_read.forecast(TEST_MONTHS)

print("SARIMA(1,0,0)(0,1,1,12), read from the ACF and PACF plots")
print(f"  AIC:      {hand_read.aic:.1f}")
print(f"  Test MAE: {mean_absolute_error(test, hand_read_forecast):.2f} °C")
print()
print(f"  Seasonal naive baseline: {SEASONAL_NAIVE_MAE:.2f} °C")
print(f"  Holt-Winters (B01):      {HOLT_WINTERS_MAE:.2f} °C")

1.64 °C. Better than the baseline, though not yet as good as Holt-Winters, and arrived at by looking at
two plots and applying two rules.

That is a respectable result for a method with no search in it at all. It is also where the classical
workflow stops and the modern one begins.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")
ax.plot(hand_read_forecast, color="seagreen", linewidth=1.5, linestyle="--",
        label="SARIMA(1,0,0)(0,1,1,12)")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("A SARIMA model read off the correlation plots", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Searching-for-the-Orders">6. Searching for the Orders</h3>
</div>

Reading plots is a skill worth having, but it is subjective and it does not scale to a hundred series.
The practical alternative is to fit every combination in a sensible range and rank them by AIC, which is
exactly what tools marketed as **auto-ARIMA** do underneath.

We can write it in a dozen lines. The differencing orders are fixed at what the tests told us in section
2 ($d = 0$, $D = 1$), and only $p$, $q$, $P$ and $Q$ are searched.

> **This cell takes a minute or two.** It fits 36 models on 1,700 observations. Some combinations will
> fail to converge; the search skips them, which is normal and not a cause for concern.

In [ ]:
def search_orders(train, p_values, q_values, P_values, Q_values, d=0, D=1, m=12):
    """Fit every combination and rank by AIC. Combinations that fail are skipped."""
    results = []

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for p, q, P, Q in itertools.product(p_values, q_values, P_values, Q_values):
            order, seasonal_order = (p, d, q), (P, D, Q, m)
            try:
                model = ARIMA(train, order=order, seasonal_order=seasonal_order).fit()
            except Exception:
                continue
            results.append({
                "order": order,
                "seasonal_order": seasonal_order,
                "AIC": model.aic,
                "BIC": model.bic,
                "model": model,
            })

    return pd.DataFrame(results).sort_values("AIC").reset_index(drop=True)


search = search_orders(train, range(3), range(3), range(2), range(2))

print(f"{len(search)} models fitted")
search.drop(columns="model").head(8).round(1)

In [ ]:
# Score the top candidates on the held-out data they have never seen
top = search.head(6).copy()
top["Test MAE"] = [
    mean_absolute_error(test, row.model.forecast(TEST_MONTHS)) for row in top.itertuples()
]

top[["order", "seasonal_order", "AIC", "BIC", "Test MAE"]].round(2)

Two things to take from this table, and they pull in different directions.

**The search agrees with the plots where it matters.** Every one of the top models has the seasonal order
`(0,1,1,12)` or `(1,1,1,12)`, confirming the seasonal MA(1) structure we read off the ACF. The plots got
the important part right; the search refines the short-lag orders from AR(1) to something slightly
richer.

**AIC cannot rank finely within the top group.** The best model by AIC is not the best on held-out data,
and the AIC spread across the top six is about 13 points while their test errors differ by barely 0.05
°C. This is Notebook A06's warning arriving on schedule: differences this small are inside the noise of a
single split, and choosing between these six on either number alone is guesswork.

What AIC *does* do reliably is separate the good models from the bad. That is the job it should be given:
narrow the field, then evaluate properly.

In [ ]:
best = search.loc[0, "model"]
best_forecast = best.forecast(TEST_MONTHS)

print(f"Chosen: SARIMA{search.loc[0, 'order']}{search.loc[0, 'seasonal_order']}")
print(f"  Test MAE: {mean_absolute_error(test, best_forecast):.2f} °C")
print()
print(f"  Hand-read SARIMA:        {mean_absolute_error(test, hand_read_forecast):.2f} °C")
print(f"  Holt-Winters (B01):      {HOLT_WINTERS_MAE:.2f} °C")
print(f"  Seasonal naive baseline: {SEASONAL_NAIVE_MAE:.2f} °C")

The searched model lands at 1.31 °C, close behind Holt-Winters at 1.22 °C and comfortably ahead of the
baseline. Two quite different families, given the same series, arrive at essentially the same accuracy.

That is a common and slightly deflating finding. Once a model captures the structure that is genuinely
there, the remaining differences are small, and they are dominated by the noise floor we measured in
Notebook A06.

**Exercise.** Run the diagnostics from Notebook A06 on the chosen model's residuals (`best.resid`): plot the ACF and run the Ljung-Box test. Compare with what the seasonal naive residuals looked like. Has SARIMA extracted the structure the baseline left behind?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-SARIMAX:-Using-What-You-Already-Know">7. SARIMAX: Using What You Already Know</h3>
</div>

Everything so far has used only the series' own history. But you often know things about the future that
the past cannot tell you: a promotion is scheduled, a public holiday is coming, a price is changing.

**SARIMAX** adds those as **exogenous** variables, entering the model as ordinary regression terms
alongside the ARIMA structure. The requirement is strict and easy to overlook: you must supply the
exogenous values **for the forecast period too**. That limits you to things you genuinely know in
advance, such as a calendar, rather than things you would also have to forecast.

The Rossmann data from Notebook A04 is a natural fit: daily sales, with a `Promo` flag that the retailer
sets in advance.

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)

store = (
    sales[sales["Store"] == 1]
    .set_index("Date")
    .sort_index()
    .asfreq("D")                      # daily, including the days the shop was shut
)

FORECAST_DAYS = 28

target = store["Sales"].astype(float)
target_train, target_test = target.iloc[:-FORECAST_DAYS], target.iloc[-FORECAST_DAYS:]

print(f"{len(store)} days, {store.index.min().date()} to {store.index.max().date()}")
print(f"Forecasting the final {FORECAST_DAYS} days")

In [ ]:
def fit_with_exog(columns, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7)):
    """Fit a SARIMA(X) model using the given exogenous columns, and score it."""
    exog = store[columns].astype(float) if columns else None
    exog_train = exog.iloc[:-FORECAST_DAYS] if columns else None
    exog_test = exog.iloc[-FORECAST_DAYS:] if columns else None

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = ARIMA(
            target_train, exog=exog_train, order=order, seasonal_order=seasonal_order
        ).fit()

    forecast = model.forecast(FORECAST_DAYS, exog=exog_test)

    return {
        "Exogenous": ", ".join(columns) if columns else "none",
        "AIC": model.aic,
        "Test MAE": mean_absolute_error(target_test, forecast),
        "coefficients": {column: model.params[column] for column in columns},
    }


first = fit_with_exog(["Promo"])
print(f"Promo coefficient: {first['coefficients']['Promo']:,.0f}")
print(f"AIC: {first['AIC']:,.1f}   Test MAE: {first['Test MAE']:.1f}")

A promotion is worth about 1,900 in extra sales, says the model. That should make you suspicious, because
Notebook A04 measured the promotion effect directly on this store and found roughly 1,000.

The discrepancy has a cause, and it is a mistake worth learning to spot.

In [ ]:
print("Promotions running while the shop was closed:",
      int(((store["Open"] == 0) & (store["Promo"] == 1)).sum()))
print("Closed days in total:                        ",
      int((store["Open"] == 0).sum()))
print(f"Correlation between Promo and Open:           {store['Promo'].corr(store['Open']):.3f}")

Promotions almost never run on a day the shop is shut, so `Promo` is partly acting as a stand-in for "the
shop was open at all". The coefficient is not measuring the promotion; it is measuring the promotion plus
a share of the difference between an open day and a closed one.

The fix is to give the model the variable it is really missing.

In [ ]:
comparisons = [
    fit_with_exog([]),
    fit_with_exog(["Promo"]),
    fit_with_exog(["Open", "Promo"]),
    fit_with_exog(["Open", "Promo", "SchoolHoliday"]),
]

summary = pd.DataFrame([
    {
        "Exogenous": result["Exogenous"],
        "AIC": result["AIC"],
        "Test MAE": result["Test MAE"],
        **{f"beta({name})": value for name, value in result["coefficients"].items()},
    }
    for result in comparisons
])

summary.round(1)

With `Open` in the model, the `Promo` coefficient falls from about 1,900 to **958**, which is within a
rounding error of the ~1,000 measured directly in Notebook A04. Controlling for the confounder recovers
the honest estimate, and the forecast improves sharply at the same time: 535 to 266 in MAE, with AIC
falling by more than 1,200.

`SchoolHoliday` makes a subtler point. Its coefficient is about 25, on sales averaging several thousand,
so its practical effect is close to nothing. Yet AIC still prefers the model that includes it, and the
held-out error improves by a couple of percent. Information criteria reward any genuine improvement in
fit; they do not ask whether the improvement is large enough to care about. That question stays yours.

Look also at what happened to `Promo` in that last row: it moved from 958 to 1,225 simply because another
correlated variable joined the model. Coefficients in a regression are not independent measurements of
separate effects, and a number that shifts this much when a neighbour is added should not be quoted as
"the value of a promotion" without qualification.

Two habits follow. **Interpret the coefficients**, because a value that disagrees with what you know
about the business is a bug report, as the first attempt here was. And **think about what each variable
is standing in for** before concluding that it causes anything.

**Exercise.** Add the day-of-week as exogenous dummy variables (`pd.get_dummies(store.index.dayofweek)`) and drop the seasonal order to `(0, 0, 0, 0)`. Does explicit calendar information do the same job as the weekly seasonal terms, and which gives the better AIC?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-Where-ARIMA-Fits">8. Where ARIMA Fits</h3>
</div>

The family, in one place:

| Model | What it adds | When |
|---|---|---|
| **AR(p)** | Past values | Series correlated with its own recent past |
| **MA(q)** | Past errors | Shocks that echo for a few periods |
| **ARMA(p,q)** | Both | Stationary series with short-range structure |
| **ARIMA(p,d,q)** | Differencing | A trend, or any non-stationarity |
| **SARIMA(...)(P,D,Q,m)** | Seasonal terms | A repeating cycle of known length |
| **SARIMAX** | Exogenous variables | You know something about the future |

The workflow rarely varies:

1. **Test for stationarity** with ADF and KPSS. Difference until both agree, and no further.
2. **Read the ACF and PACF** of the stationary series for a starting point.
3. **Search** a small grid around it, ranking by AIC.
4. **Evaluate** the shortlist on held-out data, against a baseline.
5. **Check the residuals**. Structure left behind means a better model exists.

**ARIMA against exponential smoothing.** On this series they finished level, 1.31 against 1.22, and that
is typical. The real differences are elsewhere. Exponential smoothing is easier to explain and has fewer
ways to go wrong. ARIMA is more flexible, extends naturally to exogenous variables, and rests on a
statistical foundation that yields honest prediction intervals. Knowing both, and knowing they usually
agree, is more useful than a strong opinion about which is better.

---

Both families so far assume you can name the structure in advance: a season of one fixed length, a
correlation that dies away within a few lags. The next notebook covers the models that handle what those
assumptions leave out, including several seasonal periods at once, and a trend that bends:
[B03 - Advanced Statistical Methods](./B03_Advanced_statistical_models.ipynb).